# EDA 
This notebook explores the data evailable transit data for further development


## Packages

In [ ]:
import os 
from pathlib import Path
from google.transit import gtfs_realtime_pb2
import requests
from requests.exceptions import Timeout, ConnectionError, HTTPError
from time import sleep
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
from typing import Optional, List
import mysql.connector
import geopandas as gpd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from tqdm import tqdm
from sklearn.metrics import mean_absolute_error

## Constants

In [ ]:

REALTIME_VEHICLE_UPDATES_URL = (
    "https://bct.tmix.se/gtfs-realtime/vehicleupdates.pb?operatorIds=47"
)

dir_root = Path.cwd().resolve().parent
dir_data = dir_root / "data"
dir_src = dir_root / "src"
dir_visuals = dir_root / "notebooks" / "visuals"

dir_data.mkdir(parents=True, exist_ok=True)
dir_visuals.mkdir(parents=True, exist_ok=True)

path_df = dir_data / "vehicle_updates.csv"
path_df_bus_stop = dir_data / "Kelowna_Regional_Transit_System_stops.csv"

## Functions 

In [ ]:
def get_trip_updates_for_trip(df_vu_trip: pd.DataFrame, df_trip_updates: pd.DataFrame, offset: pd.Timedelta = None) -> pd.DataFrame:
    """
    Assigns the latest trip updates to each row in the vehicle updates DataFrame for a specific trip.
    Assigns in such a way that the trip updates are assigned to the vehicle updates that have a 
    feed_timestamp greater than or equal to the feed_timestamp of the trip update minus the offset. This ensures
    that vehicle updates does not get assigned future trip updates. 

    Parameters:
    -----------
    df_vu_trip: pd.DataFrame
        DataFrame containing vehicle updates for a specific trip
    df_trip_updates: pd.DataFrame
        DataFrame containing trip updates for a specific trip
    offset: pd.Timedelta, optional
        Time offset to ensure vehicle updates do not get assigned future trip updates

    Returns:
    --------
    pd.DataFrame
        DataFrame containing vehicle updates for a specific trip with the latest trip updates assigned to each row

    """
    df_vu_trip = df_vu_trip.copy()
    df_vu_trip['arrival_delay'] = None
    df_vu_trip['departure_delay'] = None
    df_vu_trip['arrival_time'] = None
    df_vu_trip['departure_time'] = None
    for i in range(len(df_vu_trip)):
        row = df_vu_trip.iloc[i]
        trip_id = row['trip_id']
        stop_id = row['stop_id']
        feed_timestamp = row['feed_timestamp']

        if offset is not None:
            df_update = df_trip_updates[df_trip_updates['feed_timestamp'] <= feed_timestamp - offset]
        else:
            df_update = df_trip_updates[df_trip_updates['feed_timestamp'] <= feed_timestamp]
            
        df_update = df_update[(df_update['trip_id']==trip_id) & (df_update['stop_id']==stop_id)]
        if not df_update.empty:
            # the most recent trip update is near the top 
            arrival_delay = df_update.iloc[0]['arrival_delay']
            departure_delay = df_update.iloc[0]['departure_delay']
            arrival_time = df_update.iloc[0]['arrival_time']
            departure_time = df_update.iloc[0]['departure_time']
            df_vu_trip.at[df_vu_trip.index[i], 'arrival_delay'] = arrival_delay
            df_vu_trip.at[df_vu_trip.index[i], 'departure_delay'] = departure_delay
            df_vu_trip.at[df_vu_trip.index[i], 'arrival_time'] = arrival_time
            df_vu_trip.at[df_vu_trip.index[i], 'departure_time'] = departure_time
        else:
            df_vu_trip.at[df_vu_trip.index[i], 'arrival_delay'] = np.nan
            df_vu_trip.at[df_vu_trip.index[i], 'departure_delay'] = np.nan
            df_vu_trip.at[df_vu_trip.index[i], 'arrival_time'] = np.nan
            df_vu_trip.at[df_vu_trip.index[i], 'departure_time'] = np.nan

    df_vu_trip.sort_values(['feed_timestamp'], inplace=True, ascending=True)

    return df_vu_trip

## Fetch data 

In [ ]:
conn = mysql.connector.connect(
    host="192.168.1.65",
    user="transit",
    password="transit123",
    database="transitdb",
    port=3307
)


df_vehicle_updates = pd.read_sql("""
    SELECT *
    FROM TRANSIT_VEHICLE_TABLE
    WHERE feed_timestamp >= NOW() - INTERVAL 3 HOUR
    ORDER BY id DESC
""", conn)

df_trip_updates = pd.read_sql("""
    SELECT *
    FROM TRANSIT_TRIP_TABLE
    WHERE feed_timestamp >= NOW() - INTERVAL 3 HOUR
    ORDER BY id DESC
""", conn)

conn.close()



In [ ]:
df_vehicle_updates.columns

In [ ]:

df_bus_stop = pd.read_csv(path_df_bus_stop)
rename_cols = df_bus_stop.columns.tolist()[1:]
df_bus_stop.rename(columns={col: col + "_stop" for col in rename_cols}, inplace=True)

In [ ]:

df_vehicle_updates.columns

In [ ]:
df_trip_updates.columns

In [ ]:
df_vehicle_updates.isna().sum()*100/len(df_vehicle_updates)

In [ ]:
df_trip_updates.isna().sum()*100/len(df_trip_updates)

## Visualize Stops

In [ ]:
gdf = gpd.read_file(os.path.join(dir_data, "routes","routes.shp"))
gdf_8 = gdf[gdf['route_id'] == '8-KEL']

In [ ]:
df_bus_stop

In [ ]:

fig = go.Figure()

for i in range(len(df_bus_stop)):
    lat = df_bus_stop.iloc[i]['latitude_stop']
    long = df_bus_stop.iloc[i]['longitude_stop']
    name = df_bus_stop.iloc[i]['stopname_stop']
    fig.add_trace(go.Scattermap(
        lon=[long],
        lat=[lat],
        mode='markers',
        name=name,
        marker=dict(size=10)
    ))

fig.update_layout(
    map_style="open-street-map",
    map=dict(
        center=dict(lat=df_bus_stop['latitude_stop'].mean(), 
                    lon=df_bus_stop['longitude_stop'].mean()),
        zoom=11
    ),
    showlegend=True
)

fig.write_html(dir_visuals / "stops.html")
#fig.show()

In [ ]:

fig = go.Figure()

for i in range(len(df_bus_stop)):
    lat = df_bus_stop.iloc[i]['latitude_stop']
    long = df_bus_stop.iloc[i]['longitude_stop']
    name = df_bus_stop.iloc[i]['stopname_stop']
    fig.add_trace(go.Scatter(
        x=[long],
        y=[lat],
        mode='markers',
        name=name
    ))

fig.update_layout(
    #map_style="open-street-map",
    map=dict(
    #center=dict(lat=df_bus_stop['latitude_stop'].mean(), 
    #                lon=df_bus_stop['longitude_stop'].mean()),
    zoom=11
    ),
    showlegend=True
)

fig.show()

## Where buses should be at any point in time

In [ ]:
df_stop_times = pd.read_csv("/home/abdul/Projects/transit-forecasting/data/static_data/stop_times.txt")

In [ ]:
df_stop_times['trip_id'].nunique()

In [ ]:
unique_trip_ids = df_stop_times['trip_id'].unique()

In [ ]:
# GTFS times may exceed 24:00:00, so use timedelta instead of datetime
df_stop_times["arrival_time_td"] = pd.to_timedelta(df_stop_times["arrival_time"], errors="coerce")
df_stop_times["departure_time_td"] = pd.to_timedelta(df_stop_times["departure_time"], errors="coerce")


In [ ]:
check_time = pd.to_timedelta('12:00:00', errors='coerce')

dict_new = {'trip_id': [], 'stop_id': []}
for trip in unique_trip_ids[6:]:
    df_trip = df_stop_times[df_stop_times['trip_id'] ==trip]
    # inspect if check_time within df_trip's arrival time
    min_time = df_trip['arrival_time_td'].min()
    max_time = df_trip['arrival_time_td'].max()
    if check_time >= min_time and check_time <= max_time:
        #find the nearest stop_id for the check_time
        nearest_stop_id = df_trip.iloc[(df_trip['arrival_time_td'] - check_time).abs().argsort()[:1]]['stop_id'].values[0]
        dict_new['trip_id'].append(trip)
        dict_new['stop_id'].append(nearest_stop_id)


In [ ]:
df_new = pd.DataFrame(dict_new)

In [ ]:
df_bus_stop_map = df_bus_stop.set_index('stopid')

In [ ]:
df_bus_stop_map

In [ ]:
df_new['latitude_stop'] = df_new['stop_id'].apply(lambda x: df_bus_stop_map.loc[x]['latitude_stop'] if x in df_bus_stop_map.index else None)
df_new['longitude_stop'] = df_new['stop_id'].apply(lambda x: df_bus_stop_map.loc[x]['longitude_stop'] if x in df_bus_stop_map.index else None)
df_new['stopname_stop'] = df_new['stop_id'].apply(lambda x: df_bus_stop_map.loc[x]['stopname_stop'] if x in df_bus_stop_map.index else None)

In [ ]:

fig = go.Figure()

for i in range(len(df_new)):
    lat = df_new.iloc[i]['latitude_stop']
    long = df_new.iloc[i]['longitude_stop']
    name = df_new.iloc[i]['stopname_stop']
    fig.add_trace(go.Scattermap(
        lon=[long],
        lat=[lat],
        mode='markers',
        name=name,
        marker=dict(size=15,),
    ))

fig.update_layout(
    map_style="open-street-map",
    map=dict(
        center=dict(lat=df_bus_stop['latitude_stop'].mean(), 
                    lon=df_bus_stop['longitude_stop'].mean()),
        zoom=11
    ),
    showlegend=True,
    width=1500,
    height=700
)

#fig.write_html('stops.html')
fig.show()

## Bus animation

In [ ]:
# Animate bus movement using vehicle update positions over time
# Toggle this to convert UTC feed timestamps into Vancouver local time.
convert_utc_to_vancouver = True
source_timezone = "UTC"
target_timezone = "America/Vancouver"

lat_candidates = ["vehicle_latitude", "latitude", "lat", "position_latitude"]
lon_candidates = ["vehicle_longitude", "longitude", "lon", "lng", "position_longitude"]
time_candidates = ["feed_timestamp", "timestamp", "vehicle_timestamp"]
bus_id_candidates = ["vehicle_id", "vehicle_label", "trip_id", "id"]
route_candidates = ["trip_route_id", "route_id"]


def pick_existing_column(df, candidates):
    for col in candidates:
        if col in df.columns:
            return col
    return None


lat_col = pick_existing_column(df_vehicle_updates, lat_candidates)
lon_col = pick_existing_column(df_vehicle_updates, lon_candidates)
time_col = pick_existing_column(df_vehicle_updates, time_candidates)
bus_id_col = pick_existing_column(df_vehicle_updates, bus_id_candidates)
route_col = pick_existing_column(df_vehicle_updates, route_candidates)

missing_required = [
    name
    for name, value in {
        "latitude": lat_col,
        "longitude": lon_col,
        "time": time_col,
        "bus_id": bus_id_col,
    }.items()
    if value is None
]

if missing_required:
    raise ValueError(
        "Missing required columns for animation: "
        + ", ".join(missing_required)
        + f". Available columns: {df_vehicle_updates.columns.tolist()}"
    )

anim_df = df_vehicle_updates[[bus_id_col, time_col, lat_col, lon_col] + ([route_col] if route_col else [])].copy()
anim_df[time_col] = pd.to_datetime(anim_df[time_col], errors="coerce")
anim_df = anim_df.dropna(subset=[time_col, lat_col, lon_col, bus_id_col]).sort_values(time_col)

# Build a display timestamp column so slider/hover can show local Vancouver time.
if convert_utc_to_vancouver:
    if isinstance(anim_df[time_col].dtype, pd.DatetimeTZDtype):
        anim_df["display_time"] = anim_df[time_col].dt.tz_convert(target_timezone)
    else:
        anim_df["display_time"] = (
            anim_df[time_col]
            .dt.tz_localize(source_timezone, nonexistent="NaT", ambiguous="NaT")
            .dt.tz_convert(target_timezone)
        )
else:
    anim_df["display_time"] = anim_df[time_col]

anim_df = anim_df.dropna(subset=["display_time"])

# Reduce frame count so animation remains smooth in notebooks.
anim_df["frame_time"] = anim_df["display_time"].dt.floor("30s")
frame_times = anim_df["frame_time"].dropna().sort_values().unique()

if len(frame_times) == 0:
    raise ValueError("No valid timestamps available to animate.")

frames = []
for frame_time in frame_times:
    visible = (
        anim_df[anim_df["frame_time"] <= frame_time]
        .sort_values("display_time")
        .drop_duplicates(subset=[bus_id_col], keep="last")
    )

    if route_col:
        hover_text = (
            "Bus: " + visible[bus_id_col].astype(str)
            + "<br>Route: " + visible[route_col].astype(str)
            + "<br>Updated: " + visible["display_time"].dt.strftime("%Y-%m-%d %H:%M:%S %Z")
        )
    else:
        hover_text = (
            "Bus: " + visible[bus_id_col].astype(str)
            + "<br>Updated: " + visible["display_time"].dt.strftime("%Y-%m-%d %H:%M:%S %Z")
        )

    frames.append(
        go.Frame(
            name=str(frame_time),
            data=[
                go.Scattermap(
                    lon=visible[lon_col],
                    lat=visible[lat_col],
                    mode="markers",
                    marker=dict(size=12, color="crimson", opacity=0.9),
                    text=hover_text,
                    hoverinfo="text",
                )
            ],
        )
    )

initial_data = frames[0].data

time_label = "Vancouver" if convert_utc_to_vancouver else "Original"
fig = go.Figure(data=initial_data, frames=frames)
fig.update_layout(
    title=f"Bus Locations Over Time ({time_label} Time)",
    map_style="open-street-map",
    map=dict(
        center=dict(
            lat=float(anim_df[lat_col].mean()),
            lon=float(anim_df[lon_col].mean()),
        ),
        zoom=11,
    ),
    #width=1500,
    #height=700,
    updatemenus=[
        {
            "type": "buttons",
            "showactive": False,
            "x": 0.0,
            "y": 1.05,
            "buttons": [
                {
                    "label": "Play",
                    "method": "animate",
                    "args": [
                        None,
                        {
                            "frame": {"duration": 350, "redraw": True},
                            "fromcurrent": True,
                            "transition": {"duration": 0},
                        },
                    ],
                },
                {
                    "label": "Pause",
                    "method": "animate",
                    "args": [
                        [None],
                        {
                            "frame": {"duration": 0, "redraw": True},
                            "mode": "immediate",
                            "transition": {"duration": 0},
                        },
                    ],
                },
            ],
        }
    ],
    sliders=[
        {
            "x": 0.08,
            "len": 0.9,
            "currentvalue": {"prefix": f"Frame time ({time_label}): "},
            "steps": [
                {
                    "label": pd.to_datetime(frame_name).strftime("%H:%M:%S"),
                    "method": "animate",
                    "args": [
                        [frame_name],
                        {
                            "frame": {"duration": 0, "redraw": True},
                            "mode": "immediate",
                            "transition": {"duration": 0},
                        },
                    ],
                }
                for frame_name in [frame.name for frame in frames]
            ],
        }
    ],
)

fig.write_html(dir_visuals / "bus_animation.html")
# fig.show()

## Route Visualization

In [ ]:
df_shapes = pd.read_csv(dir_data / "static_data" / "shapes.txt")
df_trips = pd.read_csv(dir_data / "static_data" / "trips.txt")
df_routes = pd.read_csv(dir_data / "static_data" / "routes.txt")

# one representative shape per route (routes can have multiple shapes/directions)
df_route_shapes = df_trips[['route_id', 'shape_id']].drop_duplicates(subset=['route_id'], keep='first')
df_route_shapes = df_route_shapes.merge(df_routes[['route_id', 'route_short_name', 'route_color']], on='route_id', how='left')


In [ ]:
fig = go.Figure()

for _, row in df_route_shapes.iterrows():
    df_shape = df_shapes[df_shapes['shape_id'] == row['shape_id']].sort_values('shape_pt_sequence')
    if df_shape.empty:
        continue

    color = f"#{row['route_color']}" if pd.notna(row['route_color']) else None
    fig.add_trace(go.Scattermap(
        lon=df_shape['shape_pt_lon'],
        lat=df_shape['shape_pt_lat'],
        mode='lines',
        name=f"Route {row['route_short_name']}",
        line=dict(width=3, color=color),
    ))

fig.update_layout(
    title="Transit Routes",
    map_style="open-street-map",
    map=dict(
        center=dict(lat=df_shapes['shape_pt_lat'].mean(),
                    lon=df_shapes['shape_pt_lon'].mean()),
        zoom=11
    ),
    showlegend=True,
    width=1500,
    height=700
)

fig.write_html(dir_visuals / "routes.html")
fig.show()


## Live Congestion Nowcasting

Individual bus ETA is hard to predict (dwell times, driver behavior, signals). A more
tractable *aggregate* quantity is a **traffic congestion index** built from bus probe
speeds: `(trip_route_id, trip_direction_id, current_stop_sequence)` acts as a stable
proxy for "where on the road network" a bus is, so we can compare each bus's current
speed to its own historical free-flow baseline at that same position, then average
across many vehicles to cancel out per-trip noise.


In [ ]:
df_vu = df_vehicle_updates.copy()
df_vu['feed_timestamp'] = pd.to_datetime(df_vu['feed_timestamp'])
df_vu = df_vu[df_vu['position_speed'].notna() & (df_vu['position_speed'] >= 0)]

segment_cols = ['trip_route_id', 'trip_direction_id', 'current_stop_sequence', 'stop_id']

# free-flow baseline speed per route/direction/stop-sequence position
df_baseline = (
    df_vu.groupby(segment_cols)['position_speed']
    .median()
    .rename('baseline_speed')
    .reset_index()
)


In [ ]:
time_bin_size = "5min"
df_vu['time_bin'] = df_vu['feed_timestamp'].dt.floor(time_bin_size)

df_segment_speed = (
    df_vu.groupby(segment_cols + ['time_bin'])['position_speed']
    .mean()
    .rename('observed_speed')
    .reset_index()
)

df_congestion = df_segment_speed.merge(df_baseline, on=segment_cols, how='left')
# drop near-stationary baselines (e.g. layovers) where the ratio is unstable
df_congestion = df_congestion[df_congestion['baseline_speed'] > 0.5]
df_congestion['congestion_index'] = (
    1 - df_congestion['observed_speed'] / df_congestion['baseline_speed']
).clip(lower=0, upper=1)


In [ ]:
# network-wide index: average congestion across all active segments per time bin
df_network_congestion = (
    df_congestion.groupby('time_bin')['congestion_index']
    .mean()
    .rename('network_congestion_index')
    .reset_index()
    .sort_values('time_bin')
)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df_network_congestion['time_bin'],
    y=df_network_congestion['network_congestion_index'],
    mode='lines+markers',
    name='Network Congestion Index',
    line=dict(color='crimson'),
))
fig.update_layout(
    title="Network-wide Congestion Index Over Time",
    xaxis_title="Time",
    yaxis_title="Congestion Index (0 = free flow, 1 = stopped)",
)
fig.show()


In [ ]:
# short-horizon nowcast: linear trend fit on the recent window, extrapolated forward
trend_window_min = 30
forecast_horizon_min = 15

df_recent = df_network_congestion.tail(trend_window_min // 5).copy()
t0 = df_recent['time_bin'].min()
df_recent['t_minutes'] = (df_recent['time_bin'] - t0).dt.total_seconds() / 60

coeffs = np.polyfit(df_recent['t_minutes'], df_recent['network_congestion_index'], deg=1)
trend = np.poly1d(coeffs)

last_t = df_recent['t_minutes'].max()
future_t = np.arange(last_t + 5, last_t + forecast_horizon_min + 1, 5)
future_times = t0 + pd.to_timedelta(future_t, unit='m')
future_values = np.clip(trend(future_t), 0, 1)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df_network_congestion['time_bin'],
    y=df_network_congestion['network_congestion_index'],
    mode='lines+markers',
    name='Observed',
    line=dict(color='crimson'),
))
fig.add_trace(go.Scatter(
    x=future_times,
    y=future_values,
    mode='lines+markers',
    name=f'Nowcast (+{forecast_horizon_min} min)',
    line=dict(color='royalblue', dash='dash'),
))
fig.update_layout(
    title="Network Congestion Index: Observed vs Short-Horizon Nowcast",
    xaxis_title="Time",
    yaxis_title="Congestion Index (0 = free flow, 1 = stopped)",
)
fig.show()


### Congestion at a Single Stop

Since `stop_id` is now part of the segment key, we can drill into the congestion index
for buses approaching one particular stop (aggregated across all routes/directions
that serve it).


In [ ]:
df_congestion

In [ ]:
df_congestion['stop_id'].value_counts().index[0]

In [ ]:
#stop_id_of_interest = df_congestion['stop_id'].value_counts().index[0]
stop_id_of_interest = '103377'

stop_name_lookup = df_bus_stop.set_index('stopid')['stopname_stop']
stop_name = stop_name_lookup.get(stop_id_of_interest, str(stop_id_of_interest))

df_stop_congestion = (
    df_congestion[df_congestion['stop_id'] == stop_id_of_interest]
    .groupby('time_bin')['congestion_index']
    .mean()
    .rename('congestion_index')
    .reset_index()
    .sort_values('time_bin')
)
# feed timestamps are UTC; convert to local Vancouver time for display
df_stop_congestion['time_bin_local'] = (
    df_stop_congestion['time_bin'].dt.tz_localize('UTC').dt.tz_convert('America/Vancouver')
)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df_stop_congestion['time_bin_local'],
    y=df_stop_congestion['congestion_index'],
    mode='lines+markers',
    name=f'Stop {stop_id_of_interest}',
    line=dict(color='darkorange'),
))
fig.update_layout(
    title=f"Congestion Index Near Stop: {stop_name} ({stop_id_of_interest})",
    xaxis_title="Time (Vancouver)",
    yaxis_title="Congestion Index (0 = free flow, 1 = stopped)",
)
fig.show()


## Road Segment Congestion

Instead of nowcasting the whole network, break Route 97 into road **segments** bounded
by consecutive stops (used here as a proxy for traffic intersections, since no
intersection dataset is available). Each segment's geometry is a slice of the route
shape between two stops. Buses traveling route 97 act as remote sensors: while a bus is
en route to a given stop, its probe speed is attributed to the segment leading into
that stop, and compared against that segment's own historical free-flow baseline.

### Fetch data

In [ ]:
conn = mysql.connector.connect(
    host="10.0.0.244",
    user="transit",
    password="transit123",
    database="transitdb",
    port=3307
)


df_vehicle_updates = pd.read_sql("""
    SELECT *
    FROM TRANSIT_VEHICLE_TABLE
    WHERE feed_timestamp >= NOW() - INTERVAL 8 DAY
    ORDER BY id DESC
""", conn)
conn.close()



In [ ]:
df_vehicle_updates.columns

In [ ]:
df_stop_times = pd.read_csv("/home/abdul/Projects/transit-forecasting/data/static_data/stop_times.txt")

### Generate Road Segments

In [ ]:
from shapely.geometry import LineString
from shapely.ops import substring

route_id_of_interest = "97-KEL"

df_stops = pd.read_csv(dir_data / "static_data" / "stops.txt")
df_stops["stop_id"] = df_stops["stop_id"].astype(str)
df_stop_times["stop_id"] = df_stop_times["stop_id"].astype(str)

# pick the trip with the most stops as the representative pattern for the route/direction
df_route_trips = df_trips[df_trips["route_id"] == route_id_of_interest]
trip_stop_counts = (
    df_stop_times[df_stop_times["trip_id"].isin(df_route_trips["trip_id"])]
    .groupby("trip_id")
    .size()
)
representative_trip_id = trip_stop_counts.idxmax()
representative_shape_id = df_route_trips.loc[
    df_route_trips["trip_id"] == representative_trip_id, "shape_id"
].iloc[0]

df_route_shape = df_shapes[df_shapes["shape_id"] == representative_shape_id].sort_values(
    "shape_pt_sequence"
)
route_line = LineString(list(zip(df_route_shape["shape_pt_lon"], df_route_shape["shape_pt_lat"])))

df_route_stops = (
    df_stop_times[df_stop_times["trip_id"] == representative_trip_id]
    .sort_values("stop_sequence")
    .merge(df_stops[["stop_id", "stop_name", "stop_lat", "stop_lon"]], on="stop_id", how="left")
    .reset_index(drop=True)
)


In [ ]:
from shapely.geometry import Point

# project each stop onto the route line to get a consistent distance-along-line
# (GTFS shape_dist_traveled units are agency-defined and don't match shapely's line units)
df_route_stops["dist_along_line"] = df_route_stops.apply(
    lambda row: route_line.project(Point(row["stop_lon"], row["stop_lat"])), axis=1
)
df_route_stops = df_route_stops.sort_values("dist_along_line").reset_index(drop=True)

segments = []
for i in range(len(df_route_stops) - 1):
    start_stop = df_route_stops.iloc[i]
    end_stop = df_route_stops.iloc[i + 1]
    segment_geom = substring(
        route_line, start_stop["dist_along_line"], end_stop["dist_along_line"]
    )
    segments.append(
        {
            "segment_id": i,
            "from_stop_id": start_stop["stop_id"],
            "from_stop_name": start_stop["stop_name"],
            "to_stop_id": end_stop["stop_id"],
            "to_stop_name": end_stop["stop_name"],
            # buses report the *next* stop they're headed to as their current stop
            "to_stop_sequence": end_stop["stop_sequence"],
            "geometry": segment_geom,
        }
    )

gdf_segments = gpd.GeoDataFrame(segments, geometry="geometry", crs="EPSG:4326")
gdf_segments


### Calc Segment Congestion

In [ ]:
df_vu_route = df_vehicle_updates.copy()
df_vu_route["feed_timestamp"] = pd.to_datetime(df_vu_route["feed_timestamp"])
df_vu_route = df_vu_route[
    (df_vu_route["trip_route_id"] == route_id_of_interest)
    & df_vu_route["position_speed"].notna()
    & (df_vu_route["position_speed"] >= 0)
]

# a bus reporting current_stop_sequence == to_stop_sequence is currently traversing that segment
df_vu_route = df_vu_route.merge(
    gdf_segments[["segment_id", "to_stop_sequence", "from_stop_name", "to_stop_name"]],
    left_on="current_stop_sequence",
    right_on="to_stop_sequence",
    how="inner",
)

# free-flow baseline speed per segment, learned from historical probe speeds
df_segment_baseline = (
    df_vu_route.groupby("segment_id")["position_speed"]
    .median()
    .rename("baseline_speed")
    .reset_index()
)

time_bin_size = "5min"
df_vu_route["time_bin"] = df_vu_route["feed_timestamp"].dt.floor(time_bin_size)

df_segment_speed = (
    df_vu_route.groupby(["segment_id", "time_bin"])["position_speed"]
    .mean()
    .rename("observed_speed")
    .reset_index()
)

df_segment_congestion = df_segment_speed.merge(df_segment_baseline, on="segment_id", how="left")
# drop near-stationary baselines (e.g. layovers) where the ratio is unstable
df_segment_congestion = df_segment_congestion[df_segment_congestion["baseline_speed"] > 0.5]
df_segment_congestion["congestion_index"] = (
    1 - df_segment_congestion["observed_speed"] / df_segment_congestion["baseline_speed"]
).clip(lower=0, upper=1)


### Plots

In [ ]:
import plotly.colors as pc

# most recent congestion reading per segment for the map snapshot
df_latest_congestion = (
    df_segment_congestion.sort_values("time_bin")
    .groupby("segment_id")
    .tail(1)
    .set_index("segment_id")["congestion_index"]
)

colorscale = pc.get_colorscale("RdYlGn_r")

fig = go.Figure()
for _, row in gdf_segments.iterrows():
    congestion = df_latest_congestion.get(row["segment_id"], np.nan)
    color = (
        pc.sample_colorscale(colorscale, [congestion])[0]
        if pd.notna(congestion)
        else "gray"
    )
    lons, lats = row["geometry"].xy
    fig.add_trace(
        go.Scattermap(
            lon=list(lons),
            lat=list(lats),
            mode="lines",
            line=dict(width=5, color=color),
            name=f"{row['from_stop_name']} -> {row['to_stop_name']}",
            hovertext=(
                f"Segment {row['segment_id']}: {row['from_stop_name']} -> {row['to_stop_name']}"
                f"<br>Congestion index: {congestion:.2f}" if pd.notna(congestion) else "No recent data"
            ),
            hoverinfo="text",
        )
    )

fig.update_layout(
    title=f"Route {route_id_of_interest} Segment Congestion (latest reading)",
    map_style="open-street-map",
    map=dict(
        center=dict(lat=route_line.centroid.y, lon=route_line.centroid.x),
        zoom=11,
    ),
    showlegend=False,
    width=1500,
    height=700,
)
fig.write_html(dir_visuals / "route_97_segment_congestion.html")
fig.show()


In [ ]:
import plotly.colors as pc

anim_window = pd.Timedelta(days=1)
max_time_bin = df_segment_congestion["time_bin"].max()
df_anim_congestion = df_segment_congestion[
    df_segment_congestion["time_bin"] >= max_time_bin - anim_window
].copy()

# segment_id x time_bin grid; forward-fill so segments with stale (but not missing) data still render
df_congestion_grid = (
    df_anim_congestion.pivot_table(index="time_bin", columns="segment_id", values="congestion_index")
    .sort_index()
    .reindex(columns=gdf_segments["segment_id"])
    .ffill()
)

colorscale = pc.get_colorscale("RdYlGn_r")


def congestion_to_color(value):
    if pd.isna(value):
        return "gray"
    return pc.sample_colorscale(colorscale, [value])[0]


frames = []
for time_bin in df_congestion_grid.index:
    row_values = df_congestion_grid.loc[time_bin]
    frame_data = []
    for _, seg in gdf_segments.iterrows():
        congestion = row_values[seg["segment_id"]]
        lons, lats = seg["geometry"].xy
        frame_data.append(
            go.Scattermap(
                lon=list(lons),
                lat=list(lats),
                mode="lines",
                line=dict(width=5, color=congestion_to_color(congestion)),
                name=f"{seg['from_stop_name']} -> {seg['to_stop_name']}",
                hovertext=(
                    f"Segment {seg['segment_id']}: {seg['from_stop_name']} -> {seg['to_stop_name']}"
                    + (
                        f"<br>Congestion index: {congestion:.2f}"
                        if pd.notna(congestion)
                        else "<br>No data yet"
                    )
                ),
                hoverinfo="text",
            )
        )
    frames.append(go.Frame(name=str(time_bin), data=frame_data))

initial_data = frames[0].data

fig = go.Figure(data=initial_data, frames=frames)
fig.update_layout(
    title=f"Route {route_id_of_interest} Segment Congestion Over Time (last {anim_window})",
    map_style="open-street-map",
    map=dict(
        center=dict(lat=route_line.centroid.y, lon=route_line.centroid.x),
        zoom=11,
    ),
    showlegend=False,
    width=1500,
    height=700,
    updatemenus=[
        {
            "type": "buttons",
            "showactive": False,
            "x": 0.0,
            "y": 1.05,
            "buttons": [
                {
                    "label": "Play",
                    "method": "animate",
                    "args": [
                        None,
                        {
                            "frame": {"duration": 250, "redraw": True},
                            "fromcurrent": True,
                            "transition": {"duration": 0},
                        },
                    ],
                },
                {
                    "label": "Pause",
                    "method": "animate",
                    "args": [
                        [None],
                        {
                            "frame": {"duration": 0, "redraw": True},
                            "mode": "immediate",
                            "transition": {"duration": 0},
                        },
                    ],
                },
            ],
        }
    ],
    sliders=[
        {
            "x": 0.08,
            "len": 0.9,
            "currentvalue": {"prefix": "Time bin (UTC): "},
            "steps": [
                {
                    "label": pd.to_datetime(frame.name).strftime("%m-%d %H:%M"),
                    "method": "animate",
                    "args": [
                        [frame.name],
                        {
                            "frame": {"duration": 0, "redraw": True},
                            "mode": "immediate",
                            "transition": {"duration": 0},
                        },
                    ],
                }
                for frame in frames
            ],
        }
    ],
)
fig.write_html(dir_visuals / "route_97_segment_congestion_animation.html")
fig.show()


In [ ]:
import plotly.colors as pc

smoothing_window = 10  # number of 5-min bins to average over (~25 min)

# Smooth each segment independently after aligning it to the animation time grid.
df_smoothed_congestion_grid = (
    df_congestion_grid
    .rolling(window=smoothing_window, min_periods=1, center=True)
    .mean()
)

colorscale = pc.get_colorscale("RdYlGn_r")


def smoothed_congestion_to_color(value):
    if pd.isna(value):
        return "gray"
    return pc.sample_colorscale(colorscale, [value])[0]


frames = []
for time_bin in df_smoothed_congestion_grid.index:
    row_values = df_smoothed_congestion_grid.loc[time_bin]
    frame_data = []
    for _, seg in gdf_segments.iterrows():
        congestion = row_values[seg["segment_id"]]
        lons, lats = seg["geometry"].xy
        frame_data.append(
            go.Scattermap(
                lon=list(lons),
                lat=list(lats),
                mode="lines",
                line=dict(width=5, color=smoothed_congestion_to_color(congestion)),
                name=f"{seg['from_stop_name']} -> {seg['to_stop_name']}",
                hovertext=(
                    f"Segment {seg['segment_id']}: {seg['from_stop_name']} -> {seg['to_stop_name']}"
                    + (
                        f"<br>Smoothed congestion index: {congestion:.2f}"
                        if pd.notna(congestion)
                        else "<br>No data yet"
                    )
                ),
                hoverinfo="text",
            )
        )
    frames.append(go.Frame(name=str(time_bin), data=frame_data))

if not frames:
    raise ValueError("No congestion data is available to animate.")

fig = go.Figure(data=frames[0].data, frames=frames)
fig.update_layout(
    title=(
        f"Route {route_id_of_interest} Smoothed Segment Congestion Over Time "
        f"(last {anim_window}, {smoothing_window}-bin rolling mean)"
    ),
    map_style="open-street-map",
    map=dict(
        center=dict(lat=route_line.centroid.y, lon=route_line.centroid.x),
        zoom=11,
    ),
    showlegend=False,
    width=1500,
    height=700,
    updatemenus=[
        {
            "type": "buttons",
            "showactive": False,
            "x": 0.0,
            "y": 1.05,
            "buttons": [
                {
                    "label": "Play",
                    "method": "animate",
                    "args": [
                        None,
                        {
                            "frame": {"duration": 250, "redraw": True},
                            "fromcurrent": True,
                            "transition": {"duration": 0},
                        },
                    ],
                },
                {
                    "label": "Pause",
                    "method": "animate",
                    "args": [
                        [None],
                        {
                            "frame": {"duration": 0, "redraw": True},
                            "mode": "immediate",
                            "transition": {"duration": 0},
                        },
                    ],
                },
            ],
        }
    ],
    sliders=[
        {
            "x": 0.08,
            "len": 0.9,
            "currentvalue": {"prefix": "Time bin (UTC): "},
            "steps": [
                {
                    "label": pd.to_datetime(frame.name).strftime("%m-%d %H:%M"),
                    "method": "animate",
                    "args": [
                        [frame.name],
                        {
                            "frame": {"duration": 0, "redraw": True},
                            "mode": "immediate",
                            "transition": {"duration": 0},
                        },
                    ],
                }
                for frame in frames
            ],
        }
    ],
)
fig.write_html(dir_visuals / "route_97_smoothed_segment_congestion_animation.html")
fig.show()

In [ ]:
segment_id_of_interest = df_segment_congestion["segment_id"].value_counts().index[0]
segment_id_of_interest = 6
segment_row = gdf_segments.loc[gdf_segments["segment_id"] == segment_id_of_interest].iloc[0]
segment_label = f"{segment_row['from_stop_name']} -> {segment_row['to_stop_name']}"

df_single_segment = df_segment_congestion[
    df_segment_congestion["segment_id"] == segment_id_of_interest
].sort_values("time_bin")
# feed timestamps are UTC; convert to local Vancouver time for display
df_single_segment = df_single_segment.assign(
    time_bin_local=df_single_segment["time_bin"].dt.tz_localize("UTC").dt.tz_convert("America/Vancouver")
)

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=df_single_segment["time_bin_local"],
        y=df_single_segment["congestion_index"],
        mode="lines+markers",
        name=segment_label,
        line=dict(color="darkorange"),
    )
)
fig.update_layout(
    title=f"Congestion Index for Segment: {segment_label}",
    xaxis_title="Time (Vancouver)",
    yaxis_title="Congestion Index (0 = free flow, 1 = stopped)",
)
fig.show()


In [ ]:
smoothing_window = 5  # number of 5-min bins to average over (~25 min)

df_single_segment_smooth = df_single_segment.sort_values("time_bin_local").copy()
df_single_segment_smooth["congestion_index_smooth"] = (
    df_single_segment_smooth["congestion_index"]
    .rolling(window=smoothing_window, min_periods=1, center=True)
    .mean()
)

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=df_single_segment_smooth["time_bin_local"],
        y=df_single_segment_smooth["congestion_index"],
        mode="lines+markers",
        name="Raw",
        line=dict(color="lightgray"),
        opacity=0.6,
    )
)
fig.add_trace(
    go.Scatter(
        x=df_single_segment_smooth["time_bin_local"],
        y=df_single_segment_smooth["congestion_index_smooth"],
        mode="lines+markers",
        name=f"{smoothing_window}-bin rolling mean",
        line=dict(color="darkorange"),
    )
)
fig.update_layout(
    title=f"Smoothed Congestion Index for Segment: {segment_label}",
    xaxis_title="Time (Vancouver)",
    yaxis_title="Congestion Index (0 = free flow, 1 = stopped)",
)
fig.show()


In [ ]:
# number of distinct buses (probes) contributing to each time bin's congestion reading
df_probe_counts = (
    df_vu_route[df_vu_route["segment_id"] == segment_id_of_interest]
    .groupby("time_bin")["vehicle_id"]
    .nunique()
    .rename("probe_count")
    .reset_index()
    .sort_values("time_bin")
)
df_probe_counts["time_bin_local"] = (
    df_probe_counts["time_bin"].dt.tz_localize("UTC").dt.tz_convert("America/Vancouver")
)

fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    row_heights=[0.6, 0.4],
    vertical_spacing=0.08,
    subplot_titles=(
        f"Congestion Index for Segment: {segment_label}",
        "Number of Buses (Probes) Used per Time Bin",
    ),
)

fig.add_trace(
    go.Scatter(
        x=df_single_segment_smooth["time_bin_local"],
        y=df_single_segment_smooth["congestion_index"],
        mode="lines+markers",
        name="Raw",
        line=dict(color="lightgray"),
        opacity=0.6,
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=df_single_segment_smooth["time_bin_local"],
        y=df_single_segment_smooth["congestion_index_smooth"],
        mode="lines+markers",
        name=f"{smoothing_window}-bin rolling mean",
        line=dict(color="darkorange"),
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Bar(
        x=df_probe_counts["time_bin_local"],
        y=df_probe_counts["probe_count"],
        name="Probe count",
        marker=dict(color="steelblue"),
    ),
    row=2,
    col=1,
)

fig.update_yaxes(title_text="Congestion Index (0 = free flow, 1 = stopped)", row=1, col=1)
fig.update_yaxes(title_text="# Buses", row=2, col=1)
fig.update_xaxes(title_text="Time (Vancouver)", row=2, col=1)
fig.update_layout(
    title=f"Congestion Index vs. Probe Availability: {segment_label}",
    showlegend=True,
    height=700,
)
fig.show()


### Case study

In [ ]:
# Configure the inclusive time window used to download trip updates.
trip_data_start = "2026-09-15 13:00:00"
trip_data_end = "2026-09-15 19:00:00"
start_timestamp = pd.Timestamp(trip_data_start)
end_timestamp = pd.Timestamp(trip_data_end)

if start_timestamp > end_timestamp:
    raise ValueError("trip_data_start must be earlier than or equal to trip_data_end.")

conn = mysql.connector.connect(
    host="10.0.0.244",
    user="transit",
    password="transit123",
    database="transitdb",
    port=3307,
)

try:
    df_trip_updates = pd.read_sql(
        """
        SELECT *
        FROM TRANSIT_TRIP_TABLE
        WHERE feed_timestamp >= %s
          AND feed_timestamp <= %s
        ORDER BY id DESC
        """,
        conn,
        params=(start_timestamp.to_pydatetime(), end_timestamp.to_pydatetime()),
    )
finally:
    conn.close()

print(
    f"Downloaded {len(df_trip_updates):,} trip updates "
    f"from {start_timestamp} through {end_timestamp}."
)

In [ ]:
def get_trip_updates_for_trip(df_vu_trip: pd.DataFrame, df_trip_updates: pd.DataFrame, offset: pd.Timedelta = None) -> pd.DataFrame:
    """
    Assigns the latest trip updates to each row in the vehicle updates DataFrame for a specific trip.
    Assigns in such a way that the trip updates are assigned to the vehicle updates that have a 
    feed_timestamp greater than or equal to the feed_timestamp of the trip update minus the offset. This ensures
    that vehicle updates does not get assigned future trip updates. 

    Parameters:
    -----------
    df_vu_trip: pd.DataFrame
        DataFrame containing vehicle updates for a specific trip
    df_trip_updates: pd.DataFrame
        DataFrame containing trip updates for a specific trip
    offset: pd.Timedelta, optional
        Time offset to ensure vehicle updates do not get assigned future trip updates

    Returns:
    --------
    pd.DataFrame
        DataFrame containing vehicle updates for a specific trip with the latest trip updates assigned to each row

    """
    df_vu_trip = df_vu_trip.copy()
    df_vu_trip['arrival_delay'] = None
    df_vu_trip['departure_delay'] = None
    df_vu_trip['arrival_time'] = None
    df_vu_trip['departure_time'] = None
    for i in range(len(df_vu_trip)):
        row = df_vu_trip.iloc[i]
        trip_id = row['trip_id']
        stop_id = row['stop_id']
        feed_timestamp = row['feed_timestamp']

        if offset is not None:
            df_update = df_trip_updates[df_trip_updates['feed_timestamp'] <= feed_timestamp - offset]
        else:
            df_update = df_trip_updates[df_trip_updates['feed_timestamp'] <= feed_timestamp]
            
        df_update = df_update[(df_update['trip_id']==trip_id) & (df_update['stop_id']==stop_id)]
        if not df_update.empty:
            # the most recent trip update is near the top 
            arrival_delay = df_update.iloc[0]['arrival_delay']
            departure_delay = df_update.iloc[0]['departure_delay']
            arrival_time = df_update.iloc[0]['arrival_time']
            departure_time = df_update.iloc[0]['departure_time']
            df_vu_trip.at[df_vu_trip.index[i], 'arrival_delay'] = arrival_delay
            df_vu_trip.at[df_vu_trip.index[i], 'departure_delay'] = departure_delay
            df_vu_trip.at[df_vu_trip.index[i], 'arrival_time'] = arrival_time
            df_vu_trip.at[df_vu_trip.index[i], 'departure_time'] = departure_time
        else:
            df_vu_trip.at[df_vu_trip.index[i], 'arrival_delay'] = np.nan
            df_vu_trip.at[df_vu_trip.index[i], 'departure_delay'] = np.nan
            df_vu_trip.at[df_vu_trip.index[i], 'arrival_time'] = np.nan
            df_vu_trip.at[df_vu_trip.index[i], 'departure_time'] = np.nan

    df_vu_trip.sort_values(['feed_timestamp'], inplace=True, ascending=True)

    return df_vu_trip

In [ ]:
df_trip = df_vehicle_updates[df_vehicle_updates['trip_route_id'] =="97-KEL"]

In [ ]:
df_trip_filter = df_trip[(df_trip['feed_timestamp']>"2026-09-15 13:00:00") & (df_trip['feed_timestamp']<"2026-09-15 19:00:00")]

In [ ]:
df_comb = get_trip_updates_for_trip(df_trip_filter, df_trip_updates)

In [ ]:
df_trip_filter['position_speed'].hist()

In [ ]:
df_trip['position_speed'].hist()

## Route 97 Delay Forecasting Backtest

This experiment tests whether traffic congestion information that was available at a trip-update timestamp improves estimates of Route 97 stop-level arrival delay. Each historical trip-update record is a prediction event. The target is the final reported `arrival_delay` for the same trip instance and stop.

The final reported delay is a practical proxy for a realized delay, not a confirmed physical arrival. All vehicle and traffic features must have a `feed_timestamp` at or before the prediction timestamp minus an availability buffer. The visualization-only congestion series above uses global baselines and centered smoothing, so it is not used as a model feature.

In [ ]:
# Backtest configuration. Keep these independent from visualization settings above.
forecast_route_id = "97-KEL"
congestion_bin_size = "5min"
availability_buffer = pd.Timedelta(minutes=1)
congestion_ttl = pd.Timedelta(minutes=15)
congestion_lags = (1, 2, 3, 6)  # 5, 10, 15, and 30 minutes
congestion_rolling_window = 6  # trailing 30 minutes
minimum_training_days = 3
holdout_days = 1
random_seed = 42

# The target is the final reported delay for a trip instance and stop, not a confirmed arrival.
target_column = "final_arrival_delay"
trip_instance_columns = ["trip_id", "trip_start_timestamp"]

In [ ]:
backtest_history_days = 14
pull_new_data = False
vehicle_required_columns = {
    "id",
    "feed_timestamp",
    "trip_id",
    "trip_route_id",
    "trip_direction_id",
    "trip_start_timestamp",
    "position_speed",
    "current_stop_sequence",
    "stop_id",
    "vehicle_id",
}
trip_required_columns = {
    "id",
    "feed_timestamp",
    "trip_id",
    "trip_route_id",
    "trip_direction_id",
    "trip_start_timestamp",
    "stop_id",
    "stop_sequence",
    "arrival_delay",
}
if pull_new_data:
    conn = mysql.connector.connect(
        host="10.0.0.244",
        user="transit",
        password="transit123",
        database="transitdb",
        port=3307,
    )
    try:
        df_backtest_vehicle_updates = pd.read_sql(
            f"""
            SELECT *
            FROM TRANSIT_VEHICLE_TABLE
            WHERE trip_route_id = '{forecast_route_id}'
              AND feed_timestamp >= NOW() - INTERVAL {backtest_history_days} DAY
            ORDER BY feed_timestamp, id
            """,
            conn,
        )
        df_backtest_trip_updates = pd.read_sql(
            f"""
            SELECT *
            FROM TRANSIT_TRIP_TABLE
            WHERE trip_route_id = '{forecast_route_id}'
              AND feed_timestamp >= NOW() - INTERVAL {backtest_history_days} DAY
            ORDER BY feed_timestamp, id
            """,
            conn,
        )
    finally:
        conn.close()

    missing_vehicle_columns = vehicle_required_columns.difference(df_backtest_vehicle_updates.columns)
    missing_trip_columns = trip_required_columns.difference(df_backtest_trip_updates.columns)
    if missing_vehicle_columns or missing_trip_columns:
        raise ValueError(
            "Missing required columns: "
            f"vehicle={sorted(missing_vehicle_columns)}, trip={sorted(missing_trip_columns)}"
        )

    for dataframe in (df_backtest_vehicle_updates, df_backtest_trip_updates):
        dataframe["feed_timestamp"] = pd.to_datetime(dataframe["feed_timestamp"], errors="coerce")
        dataframe["trip_id"] = dataframe["trip_id"].astype("string")
        dataframe["stop_id"] = dataframe["stop_id"].astype("string")
        dataframe["trip_start_timestamp"] = dataframe["trip_start_timestamp"].astype("string")

    # Invalid timestamps cannot participate in a point-in-time prediction event.
    df_backtest_vehicle_updates = df_backtest_vehicle_updates.dropna(subset=["feed_timestamp"])
    df_backtest_trip_updates = df_backtest_trip_updates.dropna(
        subset=["feed_timestamp", "trip_id", "stop_id", "arrival_delay"]
    )
    df_backtest_trip_updates["arrival_delay"] = pd.to_numeric(
        df_backtest_trip_updates["arrival_delay"], errors="coerce"
    )
    df_backtest_trip_updates = df_backtest_trip_updates.dropna(subset=["arrival_delay"])

    print(
        {
            "vehicle_rows": len(df_backtest_vehicle_updates),
            "trip_update_rows": len(df_backtest_trip_updates),
            "vehicle_time_range": (
                df_backtest_vehicle_updates["feed_timestamp"].min(),
                df_backtest_vehicle_updates["feed_timestamp"].max(),
            ),
            "trip_update_time_range": (
                df_backtest_trip_updates["feed_timestamp"].min(),
                df_backtest_trip_updates["feed_timestamp"].max(),
            ),
            "trip_updates_missing_trip_start": int(
                df_backtest_trip_updates["trip_start_timestamp"].isna().sum()
            ),
        }
    )
    df_backtest_trip_updates.to_pickle(dir_data/"df_backtest_trip_updates.pkl")
    df_backtest_vehicle_updates.to_pickle(dir_data/"df_backtest_vehicle_updates.pkl")
else:
    df_backtest_trip_updates = pd.read_pickle(dir_data/"df_backtest_trip_updates.pkl")
    df_backtest_vehicle_updates = pd.read_pickle(dir_data/"df_backtest_vehicle_updates.pkl")

In [ ]:
def build_trip_instance_key(dataframe: pd.DataFrame) -> pd.Series:
    """Return a conservative trip-instance key for historical Route 97 records."""
    service_day = dataframe["feed_timestamp"].dt.strftime("%Y-%m-%d")
    start_time = dataframe["trip_start_timestamp"].fillna("missing-start-" + service_day)
    return dataframe["trip_id"].astype(str) + "|" + start_time.astype(str)


for dataframe in (df_backtest_vehicle_updates, df_backtest_trip_updates):
    dataframe["trip_instance_key"] = build_trip_instance_key(dataframe)

label_group_columns = ["trip_instance_key", "stop_id"]
df_prediction_events = df_backtest_trip_updates.sort_values(
    label_group_columns + ["feed_timestamp", "id"]
).copy()

# The final feed estimate is the proxy target. Collapse duplicate records from one feed
# snapshot, then use an as-of join that cannot expose same-snapshot information.
df_prediction_events[target_column] = df_prediction_events.groupby(
    label_group_columns, dropna=False
)["arrival_delay"].transform("last")
df_prediction_events["terminal_feed_timestamp"] = df_prediction_events.groupby(
    label_group_columns, dropna=False
)["feed_timestamp"].transform("last")

df_trip_update_snapshots = (
    df_prediction_events.sort_values(label_group_columns + ["feed_timestamp", "id"])
    .drop_duplicates(subset=label_group_columns + ["feed_timestamp"], keep="last")
    [["trip_instance_key", "stop_id", "feed_timestamp", "arrival_delay"]]
    .rename(
        columns={
            "feed_timestamp": "prior_delay_timestamp",
            "arrival_delay": "prior_arrival_delay",
        }
    )
)

df_prediction_events = pd.merge_asof(
    df_prediction_events.sort_values("feed_timestamp"),
    df_trip_update_snapshots.sort_values("prior_delay_timestamp"),
    left_on="feed_timestamp",
    right_on="prior_delay_timestamp",
    by=label_group_columns,
    direction="backward",
    allow_exact_matches=False,
).sort_values(label_group_columns + ["feed_timestamp", "id"])
df_prediction_events["target_lead_minutes"] = (
    df_prediction_events["terminal_feed_timestamp"] - df_prediction_events["feed_timestamp"]
).dt.total_seconds() / 60

df_prediction_events = df_prediction_events[
    (df_prediction_events["target_lead_minutes"] > 0)
    & df_prediction_events[target_column].notna()
].copy()

if df_prediction_events.empty:
    raise ValueError("No nonterminal Route 97 trip-update events are available for backtesting.")

has_prior_delay = df_prediction_events["prior_delay_timestamp"].notna()
if not (
    df_prediction_events.loc[has_prior_delay, "prior_delay_timestamp"]
    < df_prediction_events.loc[has_prior_delay, "feed_timestamp"]
).all():
    raise AssertionError("A prior delay feature is not strictly earlier than its prediction event.")

print(
    {
        "prediction_events": len(df_prediction_events),
        "unique_trip_instances": df_prediction_events["trip_instance_key"].nunique(),
        "events_with_prior_delay": int(has_prior_delay.sum()),
        "median_target_lead_minutes": df_prediction_events["target_lead_minutes"].median(),
        "fallback_trip_instance_events": int(
            df_prediction_events["trip_start_timestamp"].isna().sum()
        ),
    }
)

In [ ]:
vehicle_feature_columns = [
    "trip_instance_key",
    "feed_timestamp",
    "id",
    "trip_direction_id",
    "position_speed",
    "current_stop_sequence",
    "stop_id",
    "vehicle_id",
]
if "current_status" in df_backtest_vehicle_updates.columns:
    vehicle_feature_columns.append("current_status")

# Rename source timestamps before the as-of join so the availability invariant is visible.
df_vehicle_feature_source = df_backtest_vehicle_updates[vehicle_feature_columns].rename(
    columns={
        "feed_timestamp": "vehicle_feature_timestamp",
        "id": "vehicle_update_id",
        "stop_id": "vehicle_stop_id",
    }
)
df_vehicle_feature_source = df_vehicle_feature_source.sort_values(
    "vehicle_feature_timestamp"
)

df_prediction_events["feature_cutoff_timestamp"] = (
    df_prediction_events["feed_timestamp"] - availability_buffer
)
df_prediction_events = pd.merge_asof(
    df_prediction_events.sort_values("feature_cutoff_timestamp"),
    df_vehicle_feature_source,
    left_on="feature_cutoff_timestamp",
    right_on="vehicle_feature_timestamp",
    by="trip_instance_key",
    direction="backward",
    allow_exact_matches=True,
).sort_values(["feed_timestamp", "id"])

df_prediction_events["vehicle_state_age_minutes"] = (
    df_prediction_events["feed_timestamp"] - df_prediction_events["vehicle_feature_timestamp"]
).dt.total_seconds() / 60
df_prediction_events["vehicle_state_missing"] = df_prediction_events[
    "vehicle_feature_timestamp"
].isna()

has_vehicle_state = ~df_prediction_events["vehicle_state_missing"]
if not (
    df_prediction_events.loc[has_vehicle_state, "vehicle_feature_timestamp"]
    <= df_prediction_events.loc[has_vehicle_state, "feature_cutoff_timestamp"]
).all():
    raise AssertionError("A vehicle feature was sourced after its availability cutoff.")

print(
    {
        "events_with_vehicle_state": int(has_vehicle_state.sum()),
        "vehicle_state_coverage": has_vehicle_state.mean(),
        "median_vehicle_state_age_minutes": df_prediction_events.loc[
            has_vehicle_state, "vehicle_state_age_minutes"
        ].median(),
    }
)

In [ ]:
# This feature series is separate from the visualization series: it uses completed bins,
# trailing windows, and a baseline learned before the eligible evaluation period.
feature_baseline_end = (
    df_backtest_vehicle_updates["feed_timestamp"].min()
    + pd.Timedelta(days=minimum_training_days)
)

df_causal_vehicle_updates = df_backtest_vehicle_updates[
    df_backtest_vehicle_updates["position_speed"].notna()
    & (df_backtest_vehicle_updates["position_speed"] >= 0)
].merge(
    gdf_segments[["segment_id", "to_stop_sequence"]],
    left_on="current_stop_sequence",
    right_on="to_stop_sequence",
    how="inner",
    validate="many_to_one",
)
df_causal_vehicle_updates["time_bin"] = df_causal_vehicle_updates[
    "feed_timestamp"
].dt.floor(congestion_bin_size)
df_causal_vehicle_updates["bin_completed_timestamp"] = (
    df_causal_vehicle_updates["time_bin"] + pd.Timedelta(congestion_bin_size)
)

# One latest reading per vehicle and segment in a bin avoids overweighting fast reporters.
df_vehicle_bin_snapshots = (
    df_causal_vehicle_updates.sort_values(["segment_id", "vehicle_id", "time_bin", "feed_timestamp", "id"])
    .drop_duplicates(subset=["segment_id", "vehicle_id", "time_bin"], keep="last")
)

df_causal_segment_baseline = (
    df_vehicle_bin_snapshots[
        df_vehicle_bin_snapshots["bin_completed_timestamp"] <= feature_baseline_end
    ]
    .groupby("segment_id")["position_speed"]
    .median()
    .rename("causal_baseline_speed")
    .reset_index()
)

df_causal_segment_bins = (
    df_vehicle_bin_snapshots.groupby(["segment_id", "time_bin", "bin_completed_timestamp"])
    .agg(
        observed_speed=("position_speed", "mean"),
        congestion_probe_count=("vehicle_id", "nunique"),
    )
    .reset_index()
    .merge(df_causal_segment_baseline, on="segment_id", how="left", validate="many_to_one")
)
df_causal_segment_bins = df_causal_segment_bins[
    df_causal_segment_bins["causal_baseline_speed"] > 0.5
].copy()
df_causal_segment_bins["causal_congestion_index"] = (
    1
    - df_causal_segment_bins["observed_speed"]
    / df_causal_segment_bins["causal_baseline_speed"]
).clip(lower=0, upper=1)

df_causal_segment_bins = df_causal_segment_bins.sort_values(["segment_id", "time_bin"])
segment_groups = df_causal_segment_bins.groupby("segment_id", group_keys=False)
df_causal_segment_bins["causal_congestion_trailing_mean"] = segment_groups[
    "causal_congestion_index"
].transform(
    lambda values: values.rolling(congestion_rolling_window, min_periods=1).mean()
)
df_causal_segment_bins["causal_congestion_trailing_std"] = segment_groups[
    "causal_congestion_index"
].transform(
    lambda values: values.rolling(congestion_rolling_window, min_periods=2).std()
)
for lag in congestion_lags:
    df_causal_segment_bins[f"causal_congestion_lag_{lag * 5}m"] = segment_groups[
        "causal_congestion_index"
    ].shift(lag)

# Vehicle state determines the segment for the trip-update prediction event.
df_prediction_events = df_prediction_events.drop(columns="segment_id", errors="ignore").merge(
    gdf_segments[["segment_id", "to_stop_sequence"]],
    left_on="current_stop_sequence",
    right_on="to_stop_sequence",
    how="left",
    validate="many_to_one",
)
# Unmapped vehicle states have no traffic feature; float also matches their nullable segment key.
df_causal_segment_bins["segment_id"] = df_causal_segment_bins["segment_id"].astype(float)
df_prediction_events = pd.merge_asof(
    df_prediction_events.sort_values("feature_cutoff_timestamp"),
    df_causal_segment_bins.sort_values("bin_completed_timestamp"),
    left_on="feature_cutoff_timestamp",
    right_on="bin_completed_timestamp",
    by="segment_id",
    direction="backward",
    allow_exact_matches=True,
).sort_values(["feed_timestamp", "id"])

df_prediction_events["congestion_feature_age_minutes"] = (
    df_prediction_events["feed_timestamp"]
    - df_prediction_events["bin_completed_timestamp"]
).dt.total_seconds() / 60
df_prediction_events["congestion_feature_missing"] = (
    df_prediction_events["bin_completed_timestamp"].isna()
    | (df_prediction_events["congestion_feature_age_minutes"] > congestion_ttl.total_seconds() / 60)
)
congestion_feature_columns = [
    "causal_congestion_index",
    "causal_congestion_trailing_mean",
    "causal_congestion_trailing_std",
    "congestion_probe_count",
] + [f"causal_congestion_lag_{lag * 5}m" for lag in congestion_lags]
df_prediction_events.loc[
    df_prediction_events["congestion_feature_missing"], congestion_feature_columns
] = np.nan

has_congestion_feature = ~df_prediction_events["congestion_feature_missing"]
if not (
    df_prediction_events.loc[has_congestion_feature, "bin_completed_timestamp"]
    <= df_prediction_events.loc[has_congestion_feature, "feature_cutoff_timestamp"]
).all():
    raise AssertionError("A congestion feature includes an incomplete or future time bin.")

print(
    {
        "baseline_end": feature_baseline_end,
        "eligible_events_after_baseline": int(
            (df_prediction_events["feed_timestamp"] >= feature_baseline_end).sum()
        ),
        "congestion_feature_coverage": has_congestion_feature.mean(),
        "median_congestion_feature_age_minutes": df_prediction_events.loc[
            has_congestion_feature, "congestion_feature_age_minutes"
        ].median(),
    }
)

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor

# Keep full trip instances on one side of the time split and only train on labels that
# were finalized before the holdout began.
holdout_start = df_prediction_events["feed_timestamp"].max().floor("D") - pd.Timedelta(
    days=holdout_days
)
df_model_events = df_prediction_events[
    df_prediction_events["feed_timestamp"] >= feature_baseline_end
].copy()
df_test_events = df_model_events[
    df_model_events["feed_timestamp"] >= holdout_start
].copy()
test_trip_instances = set(df_test_events["trip_instance_key"])
df_train_events = df_model_events[
    (df_model_events["feed_timestamp"] < holdout_start)
    & (df_model_events["terminal_feed_timestamp"] < holdout_start)
    & ~df_model_events["trip_instance_key"].isin(test_trip_instances)
].copy()

if df_train_events.empty or df_test_events.empty:
    raise ValueError("Chronological holdout did not produce both training and test events.")
if set(df_train_events["trip_instance_key"]).intersection(test_trip_instances):
    raise AssertionError("A trip instance appears in both the training and holdout sets.")

# All baseline inputs are available at the decision time. target_lead_minutes and all
# terminal columns are deliberately excluded because they are only known in hindsight.
direction_column = "trip_direction_id_x"
if direction_column not in df_model_events.columns:
    direction_column = "trip_direction_id"

for dataframe in (df_train_events, df_test_events):
    dataframe["prediction_hour"] = (
        dataframe["feed_timestamp"].dt.hour + dataframe["feed_timestamp"].dt.minute / 60
    )
    dataframe["prediction_weekday"] = dataframe["feed_timestamp"].dt.dayofweek
    dataframe["target_stop_sequence_numeric"] = pd.to_numeric(
        dataframe["stop_sequence"], errors="coerce"
    )
    dataframe["current_stop_sequence_numeric"] = pd.to_numeric(
        dataframe["current_stop_sequence"], errors="coerce"
    )
    dataframe["direction_code"] = pd.to_numeric(
        dataframe[direction_column], errors="coerce"
    )

baseline_feature_columns = [
    "prior_arrival_delay",
    "position_speed",
    "current_stop_sequence_numeric",
    "target_stop_sequence_numeric",
    "direction_code",
    "prediction_hour",
    "prediction_weekday",
    "vehicle_state_age_minutes",
    "vehicle_state_missing",
]
traffic_feature_columns = congestion_feature_columns + [
    "congestion_feature_age_minutes",
    "congestion_feature_missing",
]

for dataframe in (df_train_events, df_test_events):
    dataframe[baseline_feature_columns + traffic_feature_columns] = dataframe[
        baseline_feature_columns + traffic_feature_columns
    ].apply(pd.to_numeric, errors="coerce")

X_train_baseline = df_train_events[baseline_feature_columns]
X_test_baseline = df_test_events[baseline_feature_columns]
X_train_traffic = df_train_events[baseline_feature_columns + traffic_feature_columns]
X_test_traffic = df_test_events[baseline_feature_columns + traffic_feature_columns]
y_train = df_train_events[target_column]
y_test = df_test_events[target_column]

model_parameters = {
    "loss": "absolute_error",
    "learning_rate": 0.08,
    "max_leaf_nodes": 31,
    "l2_regularization": 1.0,
    "max_iter": 100,
    "random_state": random_seed,
}
baseline_model = HistGradientBoostingRegressor(**model_parameters)
traffic_model = HistGradientBoostingRegressor(**model_parameters)
baseline_model.fit(X_train_baseline, y_train)
traffic_model.fit(X_train_traffic, y_train)

backtest_predictions = df_test_events[
    ["feed_timestamp", "trip_instance_key", "stop_id", target_column]
].copy()
backtest_predictions["baseline_prediction"] = baseline_model.predict(X_test_baseline)
backtest_predictions["traffic_prediction"] = traffic_model.predict(X_test_traffic)
backtest_predictions["baseline_absolute_error"] = (
    backtest_predictions["baseline_prediction"] - backtest_predictions[target_column]
).abs()
backtest_predictions["traffic_absolute_error"] = (
    backtest_predictions["traffic_prediction"] - backtest_predictions[target_column]
).abs()
backtest_predictions["traffic_error_improvement"] = (
    backtest_predictions["baseline_absolute_error"]
    - backtest_predictions["traffic_absolute_error"]
)

backtest_metrics = pd.DataFrame(
    {
        "model": ["baseline", "baseline_plus_causal_traffic"],
        "mae_seconds": [
            mean_absolute_error(y_test, backtest_predictions["baseline_prediction"]),
            mean_absolute_error(y_test, backtest_predictions["traffic_prediction"]),
        ],
        "median_absolute_error_seconds": [
            backtest_predictions["baseline_absolute_error"].median(),
            backtest_predictions["traffic_absolute_error"].median(),
        ],
        "p90_absolute_error_seconds": [
            backtest_predictions["baseline_absolute_error"].quantile(0.9),
            backtest_predictions["traffic_absolute_error"].quantile(0.9),
        ],
    }
)

print(
    {
        "holdout_start": holdout_start,
        "training_events": len(df_train_events),
        "holdout_events": len(df_test_events),
        "holdout_traffic_coverage": float(
            (~df_test_events["congestion_feature_missing"]).mean()
        ),
        "mean_traffic_error_improvement_seconds": backtest_predictions[
            "traffic_error_improvement"
        ].mean(),
    }
)
backtest_metrics

In [ ]:
bootstrap_iterations = 500
bootstrap_rng = np.random.default_rng(random_seed)
paired_improvements = backtest_predictions["traffic_error_improvement"].to_numpy()
bootstrap_mean_improvements = np.empty(bootstrap_iterations)
for iteration in range(bootstrap_iterations):
    sampled_indices = bootstrap_rng.integers(0, len(paired_improvements), len(paired_improvements))
    bootstrap_mean_improvements[iteration] = paired_improvements[sampled_indices].mean()

improvement_interval = np.quantile(bootstrap_mean_improvements, [0.025, 0.975])

backtest_predictions = backtest_predictions.join(
    df_test_events[
        [
            "congestion_feature_missing",
            "congestion_probe_count",
            "congestion_feature_age_minutes",
        ]
    ]
)
backtest_predictions["traffic_feature_available"] = ~backtest_predictions[
    "congestion_feature_missing"
]
backtest_predictions["probe_count_bucket"] = pd.cut(
    backtest_predictions["congestion_probe_count"],
    bins=[-np.inf, 0, 1, 2, np.inf],
    labels=["0", "1", "2", "3+"],
)

coverage_metrics = (
    backtest_predictions.groupby("traffic_feature_available", observed=True)
    .agg(
        events=(target_column, "size"),
        baseline_mae_seconds=("baseline_absolute_error", "mean"),
        traffic_mae_seconds=("traffic_absolute_error", "mean"),
        mean_traffic_error_improvement_seconds=("traffic_error_improvement", "mean"),
    )
    .reset_index()
)
probe_metrics = (
    backtest_predictions[backtest_predictions["traffic_feature_available"]]
    .groupby("probe_count_bucket", observed=True)
    .agg(
        events=(target_column, "size"),
        mean_traffic_error_improvement_seconds=("traffic_error_improvement", "mean"),
    )
    .reset_index()
)

print(
    {
        "paired_mean_improvement_seconds": paired_improvements.mean(),
        "paired_95pct_bootstrap_interval_seconds": tuple(improvement_interval),
        "interpretation": (
            "traffic improves holdout MAE" if improvement_interval[0] > 0
            else "traffic improvement is not conclusive"
        ),
    }
)
display(coverage_metrics)
display(probe_metrics)

In [ ]:
fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("Holdout Error by Model", "Paired Absolute-Error Improvement"),
)
fig.add_trace(
    go.Bar(
        x=backtest_metrics["model"],
        y=backtest_metrics["mae_seconds"],
        marker_color=["steelblue", "seagreen"],
        name="MAE",
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Histogram(
        x=backtest_predictions["traffic_error_improvement"],
        nbinsx=60,
        marker_color="darkorange",
        name="Per-event improvement",
    ),
    row=1,
    col=2,
)
fig.add_vline(x=0, line_dash="dash", line_color="black", row=1, col=2)
fig.update_yaxes(title_text="Mean absolute error (seconds)", row=1, col=1)
fig.update_xaxes(title_text="Seconds saved by causal traffic features", row=1, col=2)
fig.update_yaxes(title_text="Prediction events", row=1, col=2)
fig.update_layout(
    title="Route 97: Causal Traffic Features vs. Delay Baseline",
    showlegend=False,
    height=450,
)
fig.show()

### Initial Result

On the chronological holdout, the model with causal traffic features lowers mean absolute error relative to the operational-delay baseline. The paired bootstrap interval should remain above zero before treating the improvement as reliable. The effect is expected to be stronger when recent segment probe data is available.

This is evidence that available traffic information can improve Route 97's *final reported delay estimate*. It is not yet evidence against confirmed bus arrival time; a future version should replace the terminal-feed proxy with audited arrival events.